# Old Results Summary

This notebook loads the cached results from `data/old_results/` and presents them as DataFrames for quick comparison.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import ast
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', None)

OLD_RESULTS_DIR = "data/old_results"

## Load & Prepare Corpus Data

Reconstruct the same DataFrames used when the old results were generated.

In [2]:
# --- Load parsed caches (same as similarity_evaluation.ipynb) ---
PARSED_CACHE = "data/parsed_cache"

with open(os.path.join(PARSED_CACHE, "boun_parsed.pkl"), "rb") as f:
    boun_data = pickle.load(f)
with open(os.path.join(PARSED_CACHE, "followed_parsed.pkl"), "rb") as f:
    priority_followed_data = pickle.load(f)

print(f"BOUN: {len(boun_data)} rows, Followed: {len(priority_followed_data)} rows")

BOUN: 4608 rows, Followed: 9946 rows


In [3]:
# --- Filter to papers with abstracts & build sample sets (same seed as original) ---
boun_df_full = boun_data[boun_data['abstract'].notna() & (boun_data['abstract'].str.len() > 20)].reset_index(drop=True)
followed_df_full = priority_followed_data[priority_followed_data['abstract'].notna() & (priority_followed_data['abstract'].str.len() > 20)].reset_index(drop=True)

# Use title as the main display column
boun_df = boun_df_full.copy()
followed_df = followed_df_full.copy()

# Reconstruct the same 50-sample sets
np.random.seed(42)
sample_followed = followed_df.sample(50).reset_index(drop=True)
sample_boun = boun_df.sample(50).reset_index(drop=True)

print(f"BOUN corpus: {len(boun_df)} papers")
print(f"Followed corpus: {len(followed_df)} papers")
print(f"Sample followed (queries for 1a): {len(sample_followed)}")
print(f"Sample BOUN (queries for 1b): {len(sample_boun)}")

BOUN corpus: 3000 papers
Followed corpus: 5851 papers
Sample followed (queries for 1a): 50
Sample BOUN (queries for 1b): 50


## Load Old Results

In [4]:
# Load the most complete result files (the ones with bm25 + hybrid)
with open(os.path.join(OLD_RESULTS_DIR, "results_1a.pkl 13-25-53-394.pkl"), "rb") as f:
    results_1a = pickle.load(f)

with open(os.path.join(OLD_RESULTS_DIR, "results_1b.pkl 13-25-53-422.pkl"), "rb") as f:
    results_1b = pickle.load(f)

# Gemini results
with open(os.path.join(OLD_RESULTS_DIR, "results_1a_gemini.pkl"), "rb") as f:
    results_1a_gemini = pickle.load(f)

with open(os.path.join(OLD_RESULTS_DIR, "results_1b_gemini.pkl"), "rb") as f:
    results_1b_gemini = pickle.load(f)

# Original 1a (has gemini_embed that the newer file doesn't)
with open(os.path.join(OLD_RESULTS_DIR, "results_1a.pkl"), "rb") as f:
    results_1a_orig = pickle.load(f)

# Task 3 results
with open(os.path.join(OLD_RESULTS_DIR, "results_3a.pkl"), "rb") as f:
    results_3a = pickle.load(f)

with open(os.path.join(OLD_RESULTS_DIR, "results_3b.pkl"), "rb") as f:
    results_3b = pickle.load(f)

# Merge gemini_embed from original into main results
if 'gemini_embed' in results_1a_orig:
    results_1a['gemini_embed'] = results_1a_orig['gemini_embed']

print("Loaded results:")
print(f"  1a methods: {list(results_1a.keys())}")
print(f"  1b methods: {list(results_1b.keys())}")
print(f"  1a Gemini: {len(results_1a_gemini)} queries")
print(f"  1b Gemini: {len(results_1b_gemini)} queries")
print(f"  3a (author-based): {len(results_3a)} queries")
print(f"  3b (citation-based): {len(results_3b)} queries")

Loaded results:
  1a methods: ['tfidf', 'bm25', 'minilm', 'specter2', 'hybrid', 'gemini_embed']
  1b methods: ['tfidf', 'bm25', 'minilm', 'specter2', 'hybrid']
  1a Gemini: 50 queries
  1b Gemini: 50 queries
  3a (author-based): 20 queries
  3b (citation-based): 20 queries


## Task 1a: External → BOUN (Top-1 per method)

50 query papers from followed institutions, searching in the BOUN corpus.

In [5]:
METHODS_1A = ['tfidf', 'bm25', 'minilm', 'specter2', 'hybrid', 'gemini_embed']
METHOD_LABELS = {
    'tfidf': 'TF-IDF', 'bm25': 'BM25', 'minilm': 'MiniLM', 'specter2': 'SPECTER2',
    'hybrid': 'Hybrid (BM25+SPECTER2)', 'gemini_embed': 'Gemini Embed', 'gemini': 'Gemini'
}

def normalize_bm25(results_dict):
    """Min-max normalize BM25 scores to 0-1 range."""
    if 'bm25' not in results_dict:
        return
    all_scores = [score for query in results_dict['bm25'] for _, score in query]
    min_s, max_s = min(all_scores), max(all_scores)
    rng = max_s - min_s if max_s != min_s else 1.0
    results_dict['bm25'] = [
        [(idx, (score - min_s) / rng) for idx, score in query]
        for query in results_dict['bm25']
    ]

normalize_bm25(results_1a)
normalize_bm25(results_1b)
print("BM25 scores normalized to 0-1.")

def build_top1_df(results, query_df, corpus_df, methods, gemini_results=None):
    """Build a top-1 comparison DataFrame."""
    rows = []
    for i in range(len(query_df)):
        row = {'Query Paper': str(query_df.iloc[i]['title'])[:80]}
        for method in methods:
            if method in results:
                idx, score = results[method][i][0]
                title = str(corpus_df.iloc[idx]['title'])[:60]
                row[f'{METHOD_LABELS[method]} #1'] = f"[{score:.3f}] {title}"
            else:
                row[f'{METHOD_LABELS[method]} #1'] = "N/A"
        if gemini_results is not None:
            idx, score = gemini_results[i][0]
            title = str(corpus_df.iloc[idx]['title'])[:60]
            row['Gemini #1'] = f"[{score:.3f}] {title}"
        rows.append(row)
    return pd.DataFrame(rows)

df_1a = build_top1_df(results_1a, sample_followed, boun_df, METHODS_1A, results_1a_gemini)
print(f"Task 1a — External → BOUN: Top-1 per method ({len(df_1a)} queries)")
df_1a

BM25 scores normalized to 0-1.
Task 1a — External → BOUN: Top-1 per method (50 queries)


,Query Paper,TF-IDF #1,BM25 #1,MiniLM #1,SPECTER2 #1,Hybrid (BM25+SPECTER2) #1,Gemini Embed #1,Gemini #1
0,The PHANGS-MUSE survey,[0.110] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.554] The Histopathological and Genetic Effects of Long-Term Treat,[0.288] News sentiment and DeFi coin returns: An empirical analysis,[0.911] Dynamics of electrochemical dendritic propagation in circula,[0.032] The Histopathological and Genetic Effects of Long-Term Treat,[0.787] RAMPA: Robotic Augmented Reality for Machine Programming by,[0.200] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea
1,Unveiling the Universe with emerging cosmological probes,[0.115] Decentralizing Authentication for Mobile Networks: Opportuni,[0.435] TBSC 2018 Provisions on Design Shear Forces for Buildings wi,[0.376] TBSC 2018 Provisions on Design Shear Forces for Buildings wi,[0.882] TBSC 2018 Provisions on Design Shear Forces for Buildings wi,[0.033] TBSC 2018 Provisions on Design Shear Forces for Buildings wi,[0.751] TBSC 2018 Provisions on Design Shear Forces for Buildings wi,[0.150] Postmodern consumer identity projects on social media: Skinc
2,Information Technology–Based Tracing Strategy in Response to COVID-19 in Sou...,[0.069] Alterations in cerebral artery flow velocity acceleration pa,[0.537] TiO2@ZIF-8 hybrid as a type II heterojunction photocatalyst:,[0.344] Entrepreneurial Orientation and Innovation Performance in Fa,[0.881] TiO2@ZIF-8 hybrid as a type II heterojunction photocatalyst:,[0.033] TiO2@ZIF-8 hybrid as a type II heterojunction photocatalyst:,[0.761] TiO2@ZIF-8 hybrid as a type II heterojunction photocatalyst:,[0.350] Course and Instructor Related Factors Affecting Willingness
3,Assessing the accuracy of OpenET satellite-based evapotranspiration data to ...,[0.064] Pornography Craving Questionnaire: Adaptation and Psychometr,[0.393] Advanced deep learning–based approaches for semantic segment,[0.454] Advanced deep learning–based approaches for semantic segment,[0.901] Advanced deep learning–based approaches for semantic segment,[0.033] Advanced deep learning–based approaches for semantic segment,[0.748] Alterations in cerebral artery flow velocity acceleration pa,[0.600] Pornography Craving Questionnaire: Adaptation and Psychometr
4,Motion planning around obstacles with convex optimization,[0.141] Entrepreneurial Orientation and Innovation Performance in Fa,[0.510] Entrepreneurial Orientation and Innovation Performance in Fa,[0.494] Entrepreneurial Orientation and Innovation Performance in Fa,[0.934] Exploring the Role of Meaning-Focused and Form-Focused Instr,[0.033] Exploring the Role of Meaning-Focused and Form-Focused Instr,[0.790] Entrepreneurial Orientation and Innovation Performance in Fa,[0.650] Entrepreneurial Orientation and Innovation Performance in Fa
5,Gut Microbiota Modulate CD8 T Cell Responses to Influence Colitis-Associated...,[0.258] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.726] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.483] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.918] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.033] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea,[0.751] Dengue,[0.550] Genotype and phenotype spectrum of Charcot-Marie-Tooth disea
6,Statistical localization: From strong fragmentation to strong edge modes,[0.258] Novel Synthesis of Phosphorus-Doped Porous Carbons from Lotu,[0.663] Modelling and Evaluation of Work Zone Queues: Istanbul Case,[0.496] Novel Synthesis of Phosphorus-Doped Porous Carbons from Lotu,[0.938] Kant on the Ontological Argument for the Existence of God: W,[0.032] Modelling and Evaluation of Work Zone Queues: Istanbul Case,[0.803] Modelling and Evaluation of Work Zone Queues: Istanbul Case,[0.250] Kant on the Ontological Argument for the Existence of God: W
7,Cores that don't count,[0.108] Modelling and Evaluation of Work Zone Queues: Istanbul Case,[0.195] An Exper

## Task 1b: BOUN → External (Top-1 per method)

50 query papers from BOUN, searching in the followed-institution corpus.

In [6]:
METHODS_1B = ['tfidf', 'bm25', 'minilm', 'specter2', 'hybrid']

df_1b = build_top1_df(results_1b, sample_boun, followed_df, METHODS_1B, results_1b_gemini)
print(f"Task 1b — BOUN → External: Top-1 per method ({len(df_1b)} queries)")
df_1b.head()

Task 1b — BOUN → External: Top-1 per method (50 queries)


,Query Paper,TF-IDF #1,BM25 #1,MiniLM #1,SPECTER2 #1,Hybrid (BM25+SPECTER2) #1,Gemini #1
0,A Microfluidic Platform for Modeling Molecular Communication,"[0.078] FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABI",[0.423] 14 examples of how LLMs can transform materials science and,"[0.326] FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABI","[0.857] FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABI","[0.031] FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABI","[0.250] FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABI"
1,Aortic aneurysm evaluation by scanning acoustic microscopy and Raman spectro...,[0.236] Personalized brain circuit scores identify clinically distin,[0.292] Personalized brain circuit scores identify clinically distin,[0.374] Personalized brain circuit scores identify clinically distin,[0.878] Accurate proteome-wide missense variant effect prediction wi,[0.032] Personalized brain circuit scores identify clinically distin,[0.400] Personalized brain circuit scores identify clinically distin
2,New Ibuprofen Cystamine Salts With Improved Solubility and Anti‐Inflammatory...,[0.190] Best practices for addressing missing data through multiple,[0.275] Best practices for addressing missing data through multiple,[0.259] Best practices for addressing missing data through multiple,[0.890] Best practices for addressing missing data through multiple,[0.033] Best practices for addressing missing data through multiple,[0.350] TPU v4: An Optically Reconfigurable Supercomputer for Machin
3,A Dynamic Model for Managing Volunteer Engagement,[0.409] Extracorporeal Life Support Organization Registry Internatio,[0.350] Evaluating explainability for graph neural networks,[0.330] Generative AI at Work,[0.911] Generative AI at Work,[0.031] The Effect of Macroeconomic Uncertainty on Household Spendin,[0.200] Humans in 4D: Reconstructing and Tracking Humans with Transf
4,Surface Defect Formation and Passivation in Formamidinium Lead Triiodide (FA...,"[0.119] Multiple Myeloma, Version 2.2024, NCCN Clinical Practice Gui",[0.208] Development and future of droplet microfluidics,[0.198] DETRs with Hybrid Matching,[0.865] DETRs with Hybrid Matching,[0.032] DETRs with Hybrid Matching,[0.350] Development and future of droplet microfluidics


## Task 1a & 1b: Top-3 Detailed View (first 5 queries)

In [7]:
def build_topk_df(results, query_df, corpus_df, methods, k=3):
    """Build a top-K detailed DataFrame with one row per (query, method, rank)."""
    rows = []
    for i in range(len(query_df)):
        query_title = str(query_df.iloc[i]['title'])[:80]
        for method in methods:
            if method not in results:
                continue
            for rank, (idx, score) in enumerate(results[method][i][:k], 1):
                match_title = str(corpus_df.iloc[idx]['title'])[:80]
                rows.append({
                    'Query #': i,
                    'Query Paper': query_title,
                    'Method': METHOD_LABELS[method],
                    'Rank': rank,
                    'Score': round(float(score), 4),
                    'Match': match_title,
                    'Match Idx': int(idx)
                })
    return pd.DataFrame(rows)

df_1a_top3 = build_topk_df(results_1a, sample_followed, boun_df, METHODS_1A, k=3)
print(f"Task 1a — Top-3 detailed ({len(df_1a_top3)} rows)")
df_1a_top3[df_1a_top3['Query #'] < 5]

Task 1a — Top-3 detailed (900 rows)


,Query #,Query Paper,Method,Rank,Score,Match,Match Idx
0,0,The PHANGS-MUSE survey,TF-IDF,1,0.1104,Genotype and phenotype spectrum of Charcot-Marie-Tooth disease due to mutati...,480
1,0,The PHANGS-MUSE survey,TF-IDF,2,0.0919,Dengue,158
2,0,The PHANGS-MUSE survey,TF-IDF,3,0.0694,The Histopathological and Genetic Effects of Long-Term Treatment with High-M...,392
3,0,The PHANGS-MUSE survey,BM25,1,0.5541,The Histopathological and Genetic Effects of Long-Term Treatment with High-M...,392
4,0,The PHANGS-MUSE survey,BM25,2,0.4723,Genotype and phenotype spectrum of Charcot-Marie-Tooth disease due to mutati...,480
...,...,...,...,...,...,...,...
85,4,Motion planning around obstacles with convex optimization,Hybrid (BM25+SPECTER2),2,0.0320,Entrepreneurial Orientation and Innovation Performance in Family-Owned Turki...,390
86,4,Motion planning around obstacles with convex optimization,Hybrid (BM25+SPECTER2),3,0.0317,Comparative Genomics of Bifidobacterium animalis subsp. lactis Reveals Strai...,64
87,4,Motion planning around obstacles with convex optimization,Gemini Embed,1,0.7900,Entrepreneurial Orientation and Innovation Performance in Family-Owned Turki...,390
88,4,Motion planning around obstacles with convex optimization,Gemini Embed,2,0.7672,Exploring the Role of Meaning-Focused and Form-Focused Instruction on L2 Col...,80


In [8]:
df_1b_top3 = build_topk_df(results_1b, sample_boun, followed_df, METHODS_1B, k=3)
print(f"Task 1b — Top-3 detailed ({len(df_1b_top3)} rows)")
df_1b_top3[df_1b_top3['Query #'] < 5]

Task 1b — Top-3 detailed (750 rows)


,Query #,Query Paper,Method,Rank,Score,Match,Match Idx
0,0,A Microfluidic Platform for Modeling Molecular Communication,TF-IDF,1,0.0776,"FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABILITY OF THE DEVE...",292
1,0,A Microfluidic Platform for Modeling Molecular Communication,TF-IDF,2,0.0632,Reshares on social media amplify political news but do not detectably affect...,485
2,0,A Microfluidic Platform for Modeling Molecular Communication,TF-IDF,3,0.0598,14 examples of how LLMs can transform materials science and chemistry: a ref...,332
3,0,A Microfluidic Platform for Modeling Molecular Communication,BM25,1,0.4229,14 examples of how LLMs can transform materials science and chemistry: a ref...,332
4,0,A Microfluidic Platform for Modeling Molecular Communication,BM25,2,0.4181,Rethinking Vision Transformers for MobileNet Size and Speed,351
...,...,...,...,...,...,...,...
70,4,Surface Defect Formation and Passivation in Formamidinium Lead Triiodide (FA...,SPECTER2,2,0.8542,AI and Memory Wall,26
71,4,Surface Defect Formation and Passivation in Formamidinium Lead Triiodide (FA...,SPECTER2,3,0.8444,"FINANCIAL, SOCIAL, AND ENVIRONMENTAL DIMENSIONS OF THE STABILITY OF THE DEVE...",292
72,4,Surface Defect Formation and Passivation in Formamidinium Lead Triiodide (FA...,Hybrid (BM25+SPECTER2),1,0.0323,DETRs with Hybrid Matching,407
73,4,Surface Defect Formation and Passivation in Formamidinium Lead Triiodide (FA...,Hybrid (BM25+SPECTER2),2,0.0320,Development and future of droplet microfluidics,37


## Task 3a: Author-Based Evaluation (SPECTER2)

For each query author, check if the author's other papers are ranked highly.

In [9]:
rows_3a = []
for item in results_3a:
    target_title = str(item['target_title'])[:80]
    n_same = item['n_same_author']
    n_corpus = item['n_corpus']
    # Count how many of the top-K are by same author (True)
    top10 = item['ranked'][:10]
    hits_at_10 = sum(1 for _, is_same, _ in top10 if is_same)
    top3 = item['ranked'][:3]
    hits_at_3 = sum(1 for _, is_same, _ in top3 if is_same)
    best_idx, best_is_same, best_score = item['ranked'][0]
    rows_3a.append({
        'Author': item.get('author_name', 'N/A'),
        'Target Paper': target_title,
        'Same-Author Papers': n_same,
        'Corpus Size': n_corpus,
        'Top-1 Score': round(float(best_score), 4),
        'Top-1 Same Author': best_is_same,
        'Hits@3': f"{hits_at_3}/{min(3, n_same)}",
        'Hits@10': f"{hits_at_10}/{min(10, n_same)}",
    })

df_3a = pd.DataFrame(rows_3a)
print(f"Task 3a — Author-based evaluation ({len(df_3a)} queries)")
df_3a

Task 3a — Author-based evaluation (20 queries)


,Author,Target Paper,Same-Author Papers,Corpus Size,Top-1 Score,Top-1 Same Author,Hits@3,Hits@10
0,L. Ambroz,"Search for heavy, long-lived, charged particles with large ionisation energy...",17,67,0.2775,True,3/3,9/10
1,C. Alpigiani,Search for charged-lepton-flavour violation in Z-boson decays with the ATLAS...,52,102,0.2516,True,3/3,10/10
2,Nuri Ersoy,"Design, Manufacturing, and Analysis of a Carbon Fiber Reinforced Polymer Cra...",9,59,0.3511,True,3/3,7/9
3,Alí Emre Pusane,An Envelope-based Feature for NDA SNR Estimation,22,72,0.1080,True,3/3,10/10
4,Arman Çakar,Single Center Experience of SORD Neuropathy in Turkey (P1-1.Virtual),12,62,0.2606,True,3/3,8/10
5,Ziyadin Çakır,A geodetic exploration of the behavior of aseismic slip along the central se...,12,62,0.6751,True,3/3,9/10
6,L. Adamczyk,Performance of the ATLAS RPC detector and Level-1 muon barrel trigger at √(s...,52,102,0.3350,True,3/3,10/10
7,Shen Yin,Remaining Useful Life Prediction of Lithium-Ion Battery With Adaptive Noise ...,12,62,0.1341,False,1/3,7/10
8,Muhammet Deveci,Evaluation of bioeconomic practices within structural changes using picture ...,27,77,0.1503,True,3/3,9/10
9,Günhan Dündar,Integration of CMOS Photodiode and Capacitive Charge Pump Circuits for On Ch...,9,59,0.1552,True,2/3,6/9


## Task 3b: Citation-Based Evaluation (SPECTER2)

For each query paper, check if its cited papers are ranked highly.

In [10]:
rows_3b = []
for item in results_3b:
    target_title = str(item['target_title'])[:80]
    n_cited = item['n_cited']
    n_corpus = item['n_corpus']
    top10 = item['ranked'][:10]
    hits_at_10 = sum(1 for _, is_cited, _ in top10 if is_cited)
    top3 = item['ranked'][:3]
    hits_at_3 = sum(1 for _, is_cited, _ in top3 if is_cited)
    best_idx, best_is_cited, best_score = item['ranked'][0]
    rows_3b.append({
        'Target Paper': target_title,
        'Cited Papers': n_cited,
        'Corpus Size': n_corpus,
        'Top-1 Score': round(float(best_score), 4),
        'Top-1 Is Cited': best_is_cited,
        'Hits@3': f"{hits_at_3}/{min(3, n_cited)}",
        'Hits@10': f"{hits_at_10}/{min(10, n_cited)}",
    })

df_3b = pd.DataFrame(rows_3b)
print(f"Task 3b — Citation-based evaluation ({len(df_3b)} queries)")
df_3b

Task 3b — Citation-based evaluation (20 queries)


,Target Paper,Cited Papers,Corpus Size,Top-1 Score,Top-1 Is Cited,Hits@3,Hits@10
0,Functional polymeric coatings: thiol-maleimide ‘click’ chemistry as a powerf...,7,57,0.3395,True,3/3,7/7
1,Smart sanitary hardware for health monitoring,3,53,0.2074,True,3/3,3/3
2,pH-Responsive nanofiber buttresses as local drug delivery devices,5,55,0.2888,False,1/3,5/5
3,Dual-Functionalizable Hydrogel Coatings: Enabling Redox-Responsive Targeted ...,12,62,0.4125,True,3/3,9/10
4,Redox-responsive micellar nanoparticles using benzothiazole-disulfide termin...,3,53,0.5092,True,2/3,3/3
5,New Perspectives in Photocatalytic Water Treatment,3,53,0.2156,True,2/3,3/3
6,Search for cascade decays of charged sleptons and sneutrinos in final states...,3,53,0.1282,True,3/3,3/3
7,High-resolution <i>P</i>-wave tomography of the 1999 Izmit and Düzce earthqu...,3,53,0.4869,True,3/3,3/3
8,An Overview of the CMS High Granularity Calorimeter,3,53,0.3494,True,3/3,3/3
9,Photo‐magnetic imaging: a new functional imaging modality for more accurate ...,3,53,0.5346,True,3/3,3/3


## Average Scores Summary

In [11]:
all_methods = ['tfidf', 'bm25', 'minilm', 'specter2', 'hybrid', 'gemini_embed']
summary_rows = []

for method in all_methods:
    row = {'Method': METHOD_LABELS.get(method, method)}
    # Task 1a avg top-1 score
    if method in results_1a:
        scores = [float(results_1a[method][i][0][1]) for i in range(50)]
        row['1a Avg Top-1'] = round(np.mean(scores), 4)
    else:
        row['1a Avg Top-1'] = None
    # Task 1b avg top-1 score
    if method in results_1b:
        scores = [float(results_1b[method][i][0][1]) for i in range(50)]
        row['1b Avg Top-1'] = round(np.mean(scores), 4)
    else:
        row['1b Avg Top-1'] = None
    summary_rows.append(row)

# Add Gemini
scores_1a_gem = [float(results_1a_gemini[i][0][1]) for i in range(50)]
scores_1b_gem = [float(results_1b_gemini[i][0][1]) for i in range(50)]
summary_rows.append({
    'Method': 'Gemini',
    '1a Avg Top-1': round(np.mean(scores_1a_gem), 4),
    '1b Avg Top-1': round(np.mean(scores_1b_gem), 4),
})

df_summary = pd.DataFrame(summary_rows)
print("Average Top-1 Scores across 50 queries:")
df_summary

Average Top-1 Scores across 50 queries:


,Method,1a Avg Top-1,1b Avg Top-1
0,TF-IDF,0.1752,0.1662
1,BM25,0.3941,0.3633
2,MiniLM,0.4277,0.4000
3,SPECTER2,0.8970,0.8913
4,Hybrid (BM25+SPECTER2),0.0318,0.0319
5,Gemini Embed,0.7602,NaN
6,Gemini,0.4660,0.4090
